# POA Document Generator — Tsong Law Group

Generates Power of Attorney `.docx` files from the prepared templates.

> **Before you start:** run `python prep_all_templates.py` once to build the six template files.

In [ ]:
import subprocess, sys, os, io

for _pkg in ['python-docx']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', _pkg, '-q'])

from poa_generator import (
    generate_poa_v2_bytes,
    validate_v2,
    make_filename_v2,
)

print('Ready.')

---
## Step 1 — Template

Set the number of Intended Parents (`1` or `2`) and Agents (`1`, `2`, or `3`), then run the cell.

In [ ]:
NUM_IPS    = 1   # 1 or 2
NUM_AGENTS = 1   # 1, 2, or 3

_tpl_path = os.path.join(
    os.getcwd(),
    f'poa_template_{NUM_IPS}ip_{NUM_AGENTS}agent.docx'
)

if not os.path.exists(_tpl_path):
    raise FileNotFoundError(
        f'Template not found: {_tpl_path}\nRun prep_all_templates.py first.'
    )

with open(_tpl_path, 'rb') as _f:
    TEMPLATE_BYTES = _f.read()

print(f'Template loaded: {os.path.basename(_tpl_path)}')

---
## Step 2 — Intended Parent(s)

IP 2 fields are only used when `NUM_IPS = 2`.

In [ ]:
# ---------- Intended Parent 1 ----------
ip1_name     = 'CHENGFANG SHI'                # ALL CAPS
ip1_role     = 'Intended Father'               # 'Intended Father' or 'Intended Mother'
ip1_country  = "People's Republic of China"   # as it appears on the passport
ip1_passport = 'ED3297013'

# ---------- Intended Parent 2 (only used if NUM_IPS == 2) ----------
ip2_name     = ''
ip2_role     = 'Intended Mother'
ip2_country  = "People's Republic of China"
ip2_passport = ''

---
## Step 3 — Case Details

In [ ]:
child_last_name = 'SHI'                                  # ALL CAPS — appears as "Infant [NAME]"
surrogate_name  = 'SELENA MARIA AISPURO'                 # ALL CAPS
surrogate_dob   = 'May 22, 1996'                         # e.g., May 22, 1996
due_date        = 'December 4, 2025'                     # e.g., March 15, 2026
hospital_name   = 'Loma Linda University Medical Center'
hospital_city   = 'Loma Linda'
hospital_state  = 'California'

---
## Step 4 — Agent(s) / Attorney-in-Fact

For `agent{n}_id`, provide the **full phrase** including the ID type and number, for example:
- `"California's Driver License No. Y1234567"`
- `"People's Republic of China Passport No. ED3297013"`

Agents 2 and 3 are only used when `NUM_AGENTS` is 2 or 3.

In [ ]:
# ---------- Agent 1 ----------
agent1_name = 'JIAJIA GAO'                                 # ALL CAPS
agent1_dob  = '02/24/1986'
agent1_id   = "California's Driver License No. Y1234567"   # full phrase + number

# ---------- Agent 2 (only used if NUM_AGENTS >= 2) ----------
agent2_name = ''
agent2_dob  = ''
agent2_id   = ''

# ---------- Agent 3 (only used if NUM_AGENTS == 3) ----------
agent3_name = ''
agent3_dob  = ''
agent3_id   = ''

---
## Step 5 — Firm

In [ ]:
attorney_name = 'Xuelan Fang'
agency_name   = ''   # fertility agency — used in the output filename only

---
## Step 6 — Photos *(optional)*

Set each path to a JPG or PNG file. Leave blank (`''`) to keep the grey placeholder.

In [ ]:
PHOTO_PATHS = {
    'ip1':    '',   # IP 1 passport / ID photo
    'ip2':    '',   # IP 2 passport / ID photo  (2-IP cases only)
    'agent1': '',   # Agent 1 ID photo
    'agent2': '',   # Agent 2 ID photo
    'agent3': '',   # Agent 3 ID photo
}

PHOTOS = {}
for _person, _path in PHOTO_PATHS.items():
    if _path:
        if os.path.exists(_path):
            with open(_path, 'rb') as _f:
                PHOTOS[_person] = _f.read()
            print(f'  OK  {_person}: {os.path.basename(_path)}')
        else:
            print(f'  !!  {_person}: file not found — {_path}')

if not PHOTOS:
    print('No photos loaded — placeholders will be kept.')

---
## Step 7 — Generate

Run this cell to validate all inputs and save the document to the `Generated/` folder.

In [ ]:
info = {
    'ip1_name':        ip1_name,
    'ip1_role':        ip1_role,
    'ip1_country':     ip1_country,
    'ip1_passport':    ip1_passport,
    'ip2_name':        ip2_name,
    'ip2_role':        ip2_role,
    'ip2_country':     ip2_country,
    'ip2_passport':    ip2_passport,
    'child_last_name': child_last_name,
    'surrogate_name':  surrogate_name,
    'surrogate_dob':   surrogate_dob,
    'due_date':        due_date,
    'hospital_name':   hospital_name,
    'hospital_city':   hospital_city,
    'hospital_state':  hospital_state,
    'attorney_name':   attorney_name,
    'agency_name':     agency_name,
    'agent1_name':     agent1_name,
    'agent1_dob':      agent1_dob,
    'agent1_id':       agent1_id,
    'agent2_name':     agent2_name,
    'agent2_dob':      agent2_dob,
    'agent2_id':       agent2_id,
    'agent3_name':     agent3_name,
    'agent3_dob':      agent3_dob,
    'agent3_id':       agent3_id,
}

errors = validate_v2(NUM_IPS, NUM_AGENTS, info)
if errors:
    print('Validation errors:')
    for _e in errors:
        print(f'  - {_e}')
else:
    doc_bytes, count = generate_poa_v2_bytes(
        TEMPLATE_BYTES, NUM_IPS, NUM_AGENTS, info,
        photos=PHOTOS or None,
    )
    filename = make_filename_v2(NUM_IPS, info)

    OUTPUT_DIR = os.path.join(os.getcwd(), 'Generated')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    out_path = os.path.join(OUTPUT_DIR, filename)
    stem, ext = os.path.splitext(out_path)
    counter = 1
    while os.path.exists(out_path):
        out_path = f'{stem} ({counter}){ext}'
        counter += 1

    with open(out_path, 'wb') as _f:
        _f.write(doc_bytes)

    print(f'{count} field(s) replaced.')
    print(f'Saved: {out_path}')
    print()
    print('Review the document:')
    print('  - Agent pronoun (her/his) matches the agent gender')
    print('  - Notary acknowledgment state is correct')